# Estimating FPGA resource usage with alkaid

A trained logic network is only useful on an FPGA if it fits. This notebook shows
how to get a resource and timing estimate for a `torchlogix` model **without
installing a synthesis tool** — no Vivado, no Quartus, no license.

The tool is [alkaid](https://github.com/calad0i/alkaid), which traces a model into
its own intermediate representation (ALIR) and then runs a *surrogate cost model*
over the resulting gate-level graph. What you get out:

| quantity | meaning |
|---|---|
| `comb.cost` | estimated LUT usage (in 6-input-LUT equivalents) |
| `comb.latency` | estimated combinational delay, min and max over the outputs |
| `len(comb.ops)` | size of the traced circuit, in ALIR operations |

This takes well under a second for a small model, so it is cheap enough to put in
a training loop or a hyperparameter sweep — which is the point. Real synthesis
numbers still require real synthesis; the last section shows how to hand off to it.

## Requirements

```bash
pip install "torchlogix[alkaid]"
```

The `alkaid` extra is optional. Installing it also registers torchlogix's ALIR
tracer under alkaid's `alir_tracer.plugins` entry point, which is what makes
`framework="logic"` below resolve.

In [1]:
import os

# torch and scikit-learn each bundle their own copy of the OpenMP runtime;
# loading both in one process trips OpenMP's duplicate-runtime check. Must be
# set before torch is imported. Same workaround as tests/conftest.py.
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

from collections import Counter

import numpy as np
import torch
import torch.nn as nn

from torchlogix.layers import GroupSum, LogicDense
from torchlogix.utils import set_export_mode

from alkaid.converter import trace_model
from alkaid.trace import FVArrayInput, trace

torch.manual_seed(0)

## 1. Build a small model

Nothing special here — a two-layer logic network with a `GroupSum` classifier
head. Two sizing rules constrain the shapes:

- `LogicDense` needs `out_dim * lut_rank >= in_dim` so that every input is
  reachable by at least one LUT (`lut_rank` defaults to 2).
- `GroupSum(k)` splits its input into `k` equal groups, so the preceding layer's
  width must be divisible by `k`.

We use random weights, since resource usage depends on the *shape* of the
circuit rather than on how well it was trained. Swap in a trained
`state_dict` and everything below works identically.

In [2]:
IN_DIM = 16
N_CLASSES = 8

model = nn.Sequential(
    LogicDense(IN_DIM, 64, parametrization="raw",
               parametrization_kwargs={"weight_init": "random"}),
    LogicDense(64, 32, parametrization="raw",
               parametrization_kwargs={"weight_init": "random"}),
    GroupSum(N_CLASSES),
)
model.eval()

# Export mode swaps the differentiable relaxations used in training for the
# hard boolean ops the circuit will actually implement. Tracing without this
# would measure the training-time surrogate, not the deployed logic.
set_export_mode(model)

print(model)

Sequential(
  (0): LogicDense(
    16, 64
    weight: Parameter containing: [torch.float32 of size (64, 16)]
    parametrization: RawLUTParametrization
    connections: FixedDenseConnections, FixedDenseConnections()
    (parametrization): RawLUTParametrization()
    (connections): FixedDenseConnections()
  )
  (1): LogicDense(
    64, 32
    weight: Parameter containing: [torch.float32 of size (32, 16)]
    parametrization: RawLUTParametrization
    connections: FixedDenseConnections, FixedDenseConnections()
    (parametrization): RawLUTParametrization()
    (connections): FixedDenseConnections()
  )
  (2): GroupSum(k=8, tau=1.0)
)


## 2. Trace the model into ALIR

`FVArrayInput(shape).quantize(0, 1, 0)` declares the input as **unsigned,
1 integer bit, 0 fractional bits** — i.e. one bit per feature, which is what a
binarized logic network consumes.

Note the leading `1` in the shape. The traced circuit is combinational and
describes **one sample**; there is no batch dimension in hardware. Batching is a
software concept that only reappears when you call `predict` below.

`framework="logic"` selects torchlogix's tracer explicitly. Without it, alkaid
would infer the framework from the model's module path and pick its built-in
`torch` tracer, which cannot handle torchlogix's dynamic-shape forward passes.

In [3]:
inp = FVArrayInput((1, IN_DIM)).quantize(0, 1, 0)

inp_traced, out_traced = trace_model(model, inputs=inp, framework="logic")
comb = trace(inp_traced, out_traced)

comb

/Users/linogerlach/Projects/ML/alkaid/venv/lib/python3.13/site-packages/torch/fx/experimental/const_fold.py:353: UserWarning: Attempted to insert a get_attr Node with no underlying reference in the owning GraphModule! Call GraphModule.add_submodule to add the necessary submodule, GraphModule.add_parameter to add the necessary Parameter, or nn.Module.register_buffer to add the necessary buffer
  new_node = root_const_gm.graph.get_attr(in_node.target)


Solution([16 -> 8], cost=37.9, latency=1.0-3.23828125)

## 3. Read off the estimate

`trace()` runs alkaid's optimization pipeline by default — canonicalization,
dead-code and common-subexpression elimination, retracing, and the surrogate
cost model among them — so `comb` already carries per-operation cost and latency.

In [4]:
n_in, n_out = comb.shape
lat_min, lat_max = comb.latency

print(f"inputs           : {n_in} bits")
print(f"outputs          : {n_out} values")
print(f"ALIR operations  : {len(comb.ops)}")
print(f"estimated cost   : {comb.cost:.1f} LUT6-equivalents")
print(f"combinational delay: {lat_min:.2f} (fastest output) .. {lat_max:.2f} (slowest output)")

inputs           : 16 bits
outputs          : 8 values
ALIR operations  : 113
estimated cost   : 37.9 LUT6-equivalents
combinational delay: 1.00 (fastest output) .. 3.24 (slowest output)


The latency figure is in an abstract delay unit, not nanoseconds. Its practical
use is setting the pipeline depth: alkaid derives the stage count as
`ceil(max_latency / latency_cutoff)`, so `latency_cutoff` is a per-stage budget
expressed in these same units.

## 4. Check that the estimate describes the right circuit

An estimate for the wrong circuit is worse than no estimate. `comb.predict()`
runs the traced ALIR program with exact integer arithmetic, so it should agree
with the eval-mode PyTorch model **bit for bit** — not approximately.

In [5]:
x = torch.randint(0, 2, (8, IN_DIM)).bool()

expected = model(x).detach().numpy()
actual = comb.predict(x.numpy())

assert np.array_equal(expected, actual), "traced circuit diverges from the model"
print("traced circuit matches eval-mode output exactly")
print("sample output:", actual[0])

traced circuit matches eval-mode output exactly
sample output: [2. 0. 3. 3. 4. 2. 2. 2.]


## 5. Where the cost comes from

The total is just a sum over ALIR operations. Breaking it down by opcode shows
which part of the network dominates — useful when deciding what to shrink.

In [6]:
OPCODE_NAMES = {
    -2: "negate", -1: "input", 0: "add", 1: "subtract", 2: "relu",
    3: "quantize", 4: "add constant", 5: "define constant", 6: "mux",
    7: "multiply", 8: "lookup table", 9: "unary bitwise",
    10: "binary bitwise", 11: "shifted sum",
}

counts, costs = Counter(), Counter()
for op in comb.ops:
    counts[op.opcode] += 1
    costs[op.opcode] += op.cost

print(f"{'opcode':<18}{'count':>8}{'cost':>10}{'share':>9}")
print("-" * 45)
for opcode, cost in costs.most_common():
    name = OPCODE_NAMES.get(opcode, f"opcode {opcode}")
    share = cost / comb.cost * 100 if comb.cost else 0.0
    print(f"{name:<18}{counts[opcode]:>8}{cost:>10.2f}{share:>8.1f}%")
print("-" * 45)
print(f"{'total':<18}{len(comb.ops):>8}{comb.cost:>10.2f}{100.0:>8.1f}%")

opcode               count      cost    share
---------------------------------------------
add                     17     25.00    66.0%
binary bitwise          42     12.60    33.2%
add constant             3      0.30     0.8%
input                   16      0.00     0.0%
unary bitwise           35      0.00     0.0%
---------------------------------------------
total                  113     37.90   100.0%


Operations costing nothing are not wasted work — they are absorbed into their
consumers. Unary bitwise ops (an inverter) fold into the following LUT, and
inputs are just wires.

## 6. How cost scales with model size

Because tracing is fast, sweeping architectures is practical. This is the loop
you would put around a hyperparameter search to keep candidates inside a
resource budget.

In [7]:
def estimate(hidden_width, in_dim=IN_DIM, n_classes=N_CLASSES, seed=0):
    """Build a model of the given width and return its resource estimate."""
    torch.manual_seed(seed)
    m = nn.Sequential(
        LogicDense(in_dim, hidden_width, parametrization="raw",
                   parametrization_kwargs={"weight_init": "random"}),
        LogicDense(hidden_width, hidden_width // 2, parametrization="raw",
                   parametrization_kwargs={"weight_init": "random"}),
        GroupSum(n_classes),
    )
    m.eval()
    set_export_mode(m)

    i = FVArrayInput((1, in_dim)).quantize(0, 1, 0)
    c = trace(*trace_model(m, inputs=i, framework="logic"))
    return {"width": hidden_width, "cost": c.cost,
            "latency": c.latency[1], "ops": len(c.ops)}


results = [estimate(w) for w in (32, 64, 128, 256)]

print(f"{'width':>7}{'cost':>10}{'latency':>10}{'ops':>8}{'cost/width':>12}")
print("-" * 47)
for r in results:
    print(f"{r['width']:>7}{r['cost']:>10.1f}{r['latency']:>10.2f}"
          f"{r['ops']:>8}{r['cost'] / r['width']:>12.2f}")

  width      cost   latency     ops  cost/width
-----------------------------------------------
     32      13.8      2.09      75        0.43
     64      37.9      3.24     113        0.59
    128     104.2      4.38     221        0.81
    256     236.1      5.60     415        0.92


Cost grows slightly faster than linearly in the width while latency grows only
logarithmically — the `GroupSum` adder tree deepens by one level per doubling.
(Feed `results` to matplotlib if you would rather see the curve.)

## 7. Getting real numbers

The surrogate model is calibrated, not exact. When you need the actual figure,
generate an RTL project and run it through a real synthesis tool. `write()`
needs no Verilator or vendor tool — it only emits source.

In [8]:
from alkaid.codegen import RTLModel

rtl = RTLModel(comb, "/tmp/torchlogix_rtl", flavor="verilog", latency_cutoff=2.0)
rtl.write()

import json
with open("/tmp/torchlogix_rtl/metadata.json") as f:
    print(json.dumps(json.load(f), indent=2))

{
  "flavor": "verilog",
  "top_module": "torchlogix_rtl",
  "part_name": "xcvu13p-flga2577-2-e",
  "signal_count": 2,
  "cost": 41.6,
  "adder_size": 1,
  "carry_size": 1,
  "reg_bits": 65,
  "latency": 2,
  "clock_period": 5,
  "max_comb_delay": 3.48828125,
  "clock_uncertainty": 0.1,
  "io_delay_min": 0.2,
  "io_delay_max": 0.4
}


`metadata.json` reports the post-pipelining view: `latency` is now the number of
pipeline **stages** (`ceil(max_latency / latency_cutoff)`), and `reg_bits` counts
the registers those stages cost — a figure the combinational estimate above does
not include. Its `cost` differs slightly from `comb.cost` because pipelining and
ternary-adder fusion alter the circuit.

The generated project contains `build_vivado_prj.tcl` and
`build_quartus_prj.tcl`. After running either, parse the vendor reports with:

```bash
alkaid report /tmp/torchlogix_rtl
```

which gives measured LUT, FF, DSP, and Fmax numbers in the same table format.

## Caveats

- **The cost model targets 6-input LUTs** (Xilinx-style, `LUT_X=6`). Estimates
  for architectures with a different LUT size will be systematically off.
- **`comb.cost` covers combinational logic only** — no registers, no I/O. Use the
  `reg_bits` from the RTL metadata once you have chosen a pipeline depth.
- **Synthesis optimizes further.** Vendor tools do their own logic minimization,
  so the measured LUT count is usually below the estimate.
- **Trace what you will deploy.** `set_export_mode()` is not optional: without it
  you measure the training-time relaxation instead of the boolean circuit.
- **Export mode currently supports `lut_rank=2` only.** Wider LUTs train fine but
  cannot yet be traced, so they cannot be estimated this way.